hyperparamters and setup 
all global config lives here

In [38]:
import torch
import torch.nn as nn
from torch.nn import functional as F
#hyperparamters

batch_size = 16 #sequence processed in parallel
block_size = 64 #max context length
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3

device = 'cuda' if torch .cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 128   #embedding dimension
n_head = 8
n_layer = 6
dropout = 0.2
torch.manual_seed(1337)

loading data and tokenziation

In [40]:

import os

text = ""

for filename in os.listdir("data"):
    if filename.endswith(".txt"):
        with open(
            os.path.join("data", filename),
            "r",
            encoding="utf-8"
        ) as f:
            text += f.read() + "\n\n"

print("Characters:", len(text))
print(text[:500])

print("Characters:", len(text))
print(text[:500])

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = {i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]

decode= lambda l: ''.join([itos[i] for i in l])

#train /val split

data = torch.tensor(encode(text), dtype = torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# batch sampler

def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    x, y = x.to(device), y.to(device)

    return x, y


@torch.no_grad()
def estimate_loss():
    out = {}

    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            x, y = get_batch(split)
            logits, loss = model(x, y)
            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()

    return out

Characters: 94418776


===== HUMAN PSYCHOLOGY (Section 1) =====

Sigmund Freud proposed that much of human behavior is influenced by unconscious thoughts and desires. Researchers have found that abraham Maslow proposed a hierarchy of needs, suggesting basic needs must be met before higher growth needs are pursued. Crucially, stories and vivid images tend to be remembered far better than random isolated facts. Studies suggest that resilience is the capacity to adapt well in the face of adversity, trauma, or significa
Characters: 94418776


===== HUMAN PSYCHOLOGY (Section 1) =====

Sigmund Freud proposed that much of human behavior is influenced by unconscious thoughts and desires. Researchers have found that abraham Maslow proposed a hierarchy of needs, suggesting basic needs must be met before higher growth needs are pursued. Crucially, stories and vivid images tend to be remembered far better than random isolated facts. Studies suggest that resilience is the capacity to adapt well in 

head single attention head each token will project into three vectors: a query(what am i looking for?), a key(what do i contain?), and a value(what will i pass along?)

the attention score between two postions is q>k /root head_size

In [41]:
class Head(nn.Module):
    """one head of self attention"""

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer(
            'tril',
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v

        return out

Multiple heads run in parallel, each with head_size = n_embd // n_head. This lets the model simultaneously attend to different aspects of the context — one head might track grammar, another might track coreference, and so on.

In [42]:
class MultiHeadAttention(nn.Module):
    def __init__(self,num_heads,head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd,n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

After attention (which lets tokens communicate), each token is independently processed by a small MLP — this is where the model "thinks" per-position. The 4× expansion (64→256→64) is standard in transformers, giving the network a bottleneck-and-expand structure.

In [43]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

is is a proper decoder-only transformer. It adds two embedding tables to the stack of Blocks: a token embedding (maps each character to a vector) and a positional embedding (tells each position in the sequence where it sits).

In [44]:
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        # Token and positional embeddings
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # Transformer blocks
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head=n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Token embeddings: (B, T, C)
        tok_emb = self.token_embedding_table(idx)

        # Position embeddings: (T, C)
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=device)
        )

        # Combine token + positional embeddings
        x = tok_emb + pos_emb

        # Transformer blocks
        x = self.blocks(x)

        # Final layer norm
        x = self.ln_f(x)

        # Language model head
        logits = self.lm_head(x)

        # Compute loss if targets provided
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):

            
            # Crop to last block_size tokens
            idx_cond = idx[:, -block_size:]

            # Get predictions
            logits, _ = self(idx_cond)

            # Focus only on last time step
            logits = logits[:, -1, :]

            # Convert to probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample next token
            idx_next = torch.multinomial(probs, num_samples=1)

            # Append sampled token
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

the training loop is intentionally minimal. AdamW is used — Adam with decoupled weight decay. At each step: sample a batch, do a forward pass to get the loss, call loss.backward() to compute gradients via backpropagation, then step the optimizer.

Every eval_interval steps the model is temporarily put in eval mode and estimate_loss() averages the loss over eval_iters batches — this smooths out the noisy single-batch estimate and makes the training curve readable.

In [45]:
model = BigramLanguageModel()
m     = model.to(device)
print(sum(p.numel() for p in m.parameters()) / 1e6, 'M parameters')
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, "
              f"val loss {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss= model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward() 
    optimizer.step() 
def generate_from_prompt(prompt, max_new_tokens=200, temperature=0.8):
    idx = torch.tensor([encode(prompt)],
                       dtype=torch.long,
                       device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = m(idx_cond)

        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)

        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)

    return decode(idx[0].tolist())

1.213509 M parameters
step 0: train loss 4.4784, val loss 4.4846
step 100: train loss 2.4873, val loss 2.4712
step 200: train loss 2.3934, val loss 2.3907
step 300: train loss 2.2894, val loss 2.2936
step 400: train loss 2.1138, val loss 2.1441
step 500: train loss 1.9463, val loss 1.9972
step 600: train loss 1.7829, val loss 1.8433
step 700: train loss 1.6317, val loss 1.7239
step 800: train loss 1.5190, val loss 1.6166
step 900: train loss 1.4019, val loss 1.5321
step 1000: train loss 1.2729, val loss 1.3951
step 1100: train loss 1.1802, val loss 1.3286
step 1200: train loss 1.0903, val loss 1.2389
step 1300: train loss 1.0097, val loss 1.1630
step 1400: train loss 0.9385, val loss 1.0928
step 1500: train loss 0.8674, val loss 1.0309
step 1600: train loss 0.7871, val loss 0.9510
step 1700: train loss 0.7364, val loss 0.8918
step 1800: train loss 0.6744, val loss 0.8369
step 1900: train loss 0.6286, val loss 0.8072
step 2000: train loss 0.5878, val loss 0.7597
step 2100: train loss 0.

In [48]:
print(generate_from_prompt("time"))

time, social loafing describes the the transfer belief atter that the can Empire, and printant H meast beat between two surfaces behavior over time, and social connection, many other topics within the fir
